# Experiment A: Pretrained YOLOv8n Baseline Evaluation
### AI-Based Smart Gate Monitoring System
**Author:** Siddharth Dhanush R (Team Lead)
**Objective:** Benchmark the off-the-shelf COCO-pretrained `yolov8n.pt` on gate surveillance footage to establish reference detection accuracy and inference throughput before custom fine-tuning.

## 1. Install Dependencies & Check GPU Runtime

In [ ]:
# In Google Colab, install Ultralytics and dependencies
!pip install -q ultralytics opencv-python-headless matplotlib

In [ ]:
import torch
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Check hardware acceleration
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"[INFO] PyTorch Version: {torch.__version__}")
print(f"[INFO] Active Compute Device: {device}")
if device.startswith('cuda'):
    print(f"[INFO] GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM Allocated: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[WARNING] Running on CPU. Training will require Colab GPU runtime.")

## 2. Load Pretrained YOLOv8n Weights

In [ ]:
# Load lightweight YOLOv8 nano model (3.2M params)
model = YOLO('yolov8n.pt')
print("[INFO] Loaded pretrained YOLOv8n model.")

# Define target classes in COCO taxonomy
TARGET_CLASSES = {
    0: 'person',
    1: 'bicycle',
    2: 'car',
    3: 'motorcycle',
    5: 'bus',
    7: 'truck'
}
print(f"[INFO] Filtering to {len(TARGET_CLASSES)} gate monitoring classes: {list(TARGET_CLASSES.values())}")

## 3. Test Inference on Sample Gate Footage / Images

In [ ]:
# Use sample image or extract from uploaded gate video
sample_image_url = 'https://ultralytics.com/images/bus.jpg'  # Standard verification sample
results = model.predict(source=sample_image_url, conf=0.35, iou=0.5, classes=list(TARGET_CLASSES.keys()), device=device)

res = results[0]
print(f"[INFO] Detected {len(res.boxes)} relevant objects.")
for box in res.boxes:
    cls_id = int(box.cls[0].item())
    conf = float(box.conf[0].item())
    coords = [int(v) for v in box.xyxy[0].tolist()]
    print(f" - Class: {TARGET_CLASSES.get(cls_id, 'other')} | Conf: {conf:.4f} | BBox: {coords}")

# Display annotated output
annotated_bgr = res.plot()
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title('Baseline YOLOv8n Target Class Detections')
plt.show()

## 4. Benchmark Inference Throughput (FPS & Latency)

In [ ]:
# Benchmark 50 inference iterations
test_frame = res.orig_img
warmup = 10
iterations = 50

for _ in range(warmup):
    _ = model.predict(source=test_frame, conf=0.35, device=device, verbose=False)

latencies = []
for _ in range(iterations):
    t0 = time.perf_counter()
    _ = model.predict(source=test_frame, conf=0.35, device=device, verbose=False)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

mean_latency = np.mean(latencies)
fps = 1000.0 / mean_latency
print(f"[BENCHMARK] Baseline YOLOv8n Latency: {mean_latency:.2f} ms per frame")
print(f"[BENCHMARK] Baseline YOLOv8n Throughput: {fps:.1f} FPS")

## 5. Record Baseline Benchmark Results
Save the measured figures into `ai/experiments/experiment_log.md` and `ai/evaluation/results.csv`.